# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the 
FAIR² dataset (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
This dataset is described by a Croissant schema and can be loaded from the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

This notebook illustrates how to load the metadata, explore record sets, fields, and records, and perform initial exploratory analysis with Pandas.

In [ ]:
# Make sure mlcroissant is installed in your environment
!pip install -U mlcroissant pandas

## 1. Data Loading
Let's load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Name and description are attributes, not dictionary keys
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's enumerate all available record sets, their fields, and their `@id`s.

We'll inspect the dataset object to list:
- All available record sets (via their `@id`)
- The fields within each record set (with their `@id`, name, dataType, etc.)

In [ ]:
# List all record sets by @id, show their fields by @id as well
from typing import List

# mlcroissant exposes .record_sets as a list of RecordSet objects
print("Available record sets:")
record_set_ids = []
for rs in dataset.record_sets:
    print(f"- RecordSet name: {rs.name}\n  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - name: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      dataType: {field.data_type}")
    print()
if not record_set_ids:
    print("No record sets found in this schema. Please check the schema or contact the dataset provider.")

## 3. Data Extraction
Now, let's extract all data records for each record set, referencing them by their `@id`s. We'll load them into pandas DataFrames using the field `@id` as the column names.

You can adjust the record sets to load based on the output above.

In [ ]:
# Prepare DataFrames for each record set by @id
dataframes = {}
if not record_set_ids:
    print("No record sets available for data extraction.")
else:
    for rs_id in record_set_ids:
        print(f"Loading records for record set @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        # records are dicts keyed by field @id; so DataFrame columns will be field @id
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records. Columns:")
        print(df.columns.tolist())
        dataframes[rs_id] = df
    if dataframes:
        # Pick the first record set as an example for preview
        main_rs_id = record_set_ids[0]
        print(f"\nPreview of first 5 rows for record set: {main_rs_id}")
        display(dataframes[main_rs_id].head())
    else:
        print("No DataFrames could be constructed.")

## 4. Exploratory Data Analysis (EDA)
Now that we've loaded our data into DataFrames, let's proceed with some basic processing and exploration:
- Filtering records based on a numeric field
- Normalizing a numeric field
- Grouping by a categorical field (if available)

For demonstration, we'll locate a numeric field (such as 'Age' if present), and a group field (such as 'Sex' or raw field `@id`). All field references will use their full `@id` from the overview above.

In [ ]:
import numpy as np

# Automatically locate the first numeric field in the main record set
numeric_field_id = None
group_field_id = None
if record_set_ids:
    main_rs = [rs for rs in dataset.record_sets if rs.id == main_rs_id][0]
    df = dataframes[main_rs_id]
    # Find the first numeric field
    for field in main_rs.fields:
        # data_type could be 'Integer', 'Float', etc.
        if str(field.data_type).lower() in ['integer', 'float', 'number'] and field.id in df.columns:
            numeric_field_id = field.id
            break
    # Find a likely group field
    for field in main_rs.fields:
        if str(field.data_type).lower() == 'text' and field.id in df.columns and field.id.lower().endswith('sex'):
            group_field_id = field.id
            break
    if not numeric_field_id:
        print("No numeric field found in the main record set for EDA. Update this cell after reviewing earlier outputs.")
    else:
        # Display value distribution for the numeric field
        print(f"Basic statistics for field @id '{numeric_field_id}':")
        print(df[numeric_field_id].describe())
        # Choose a threshold at the 75th percentile as an example filter threshold
        try:
            threshold = df[numeric_field_id].dropna().quantile(0.75)
        except Exception:
            threshold = 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize the numeric field
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())
        # Try to group by a group field if available
        if group_field_id:
            print(f"\nGrouping by field @id '{group_field_id}' (likely categorical):")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            print(grouped_df)
        else:
            print("No suitable group field found to group the filtered records.")

## 5. Visualization
Let's visualize the distribution of a numeric field and the comparison across groups (if present).

We'll generate a histogram for the numeric field, and, if a group field is available, a boxplot by groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only visualize if we found a numeric field
if record_set_ids and numeric_field_id:
    fig, axs = plt.subplots(1, 2 if group_field_id else 1, figsize=(10 if group_field_id else 6, 5))
    if not group_field_id:
        axs = [axs]
    sns.histplot(df[numeric_field_id].dropna(), bins=10, ax=axs[0], kde=True)
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    axs[0].set_xlabel(numeric_field_id)
    axs[0].set_ylabel("Count")
    if group_field_id:
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id, ax=axs[1])
        axs[1].set_title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping visualization because numeric field was not determined in EDA step.")

## 6. Conclusion
In this notebook, we:
- Showed how to load Croissant datasets using the `mlcroissant` library referencing all entities by their `@id`
- Explored available record sets and fields using their `@id`s
- Loaded and previewed records for the main record set
- Performed basic EDA including filtering, normalization, and group aggregation by `@id` fields
- Visualized numeric distributions and groupings

__You can extend this notebook by exploring additional record sets or fields, using their `@id` references from the schema overview.__

For more complex analysis, reference the Croissant documentation and leverage the field and record set IDs provided to ensure semantic precision and reproducibility.

> _Last run using `mlcroissant` version:

If you have further questions about the FAIR² dataset or the Croissant schema, please refer to [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) or contact the authors.